#  Notebook 03 — SQL Analysis

## Goal
Analyze equipment failure patterns using SQL queries to:
- Understand failure distributions across machine types
- Identify sensor readings associated with failures
- Discover patterns in failure types
- Generate actionable insights for maintenance optimization

## 1.Import Libraries

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
os.chdir('..')
print(os.getcwd())

import warnings

warnings.filterwarnings('ignore')
print("All Libraries Imported Successfully")

/Users/jayeshranghera/Documents/Projects/predictive-maintenance-analysis
All Libraries Imported Successfully


In [2]:
#connect to SQlite database
db_path = 'data/database/predictive_maintenance.db'
conn = sqlite3.connect(db_path)

print(f'Connected to Database: {db_path}')
print("Database Connetion Established")

Connected to Database: data/database/predictive_maintenance.db
Database Connetion Established


## 2.Basic Data Exploration

In [3]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

             name
0  equipment_data


In [4]:
#query 1 - total record count
query = "SELECT COUNT(*) as total_records FROM equipment_data;"

result = pd.read_sql(query, conn)
print('Total Records in Database:')
print(result)

Total Records in Database:
   total_records
0          10000


In [5]:
#query 2 - machine type distribution
query = """SELECT 
    machine_type,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM equipment_data), 2) as percentage
FROM equipment_data
GROUP BY machine_type
ORDER BY count DESC;"""

result = pd.read_sql(query,conn)
print('Machine Type Distribution')
print(result.to_string(index=False))

Machine Type Distribution
machine_type  count  percentage
           L   6000       60.00
           M   2997       29.97
           H   1003       10.03


In [6]:
#query 3 - overall failure rate
query = """
SELECT 
    SUM(CASE WHEN machine_failure = 0 THEN 1 ELSE 0 END) as no_failure,
    SUM(CASE WHEN machine_failure = 1 THEN 1 ELSE 0 END) as failure,
    ROUND(AVG(machine_failure) * 100, 2) as failure_rate_pct
FROM equipment_data;
"""

result = pd.read_sql(query, conn)
print('Overall Failure Statistics:\n')
print(result.to_string(index=False))

Overall Failure Statistics:

 no_failure  failure  failure_rate_pct
       9661      339              3.39



## 3. Failure Rate Analysis

Analyzing failure rates across different machine types to identify which types are most prone to failures.

In [7]:
#query 4 - failure rate by machine type
query = """SELECT 
    machine_type,
    COUNT(*) as total_machines,
    SUM(machine_failure) as total_failures,
    ROUND(AVG(machine_failure) * 100, 2) as failure_rate_pct
FROM equipment_data
GROUP BY machine_type
ORDER BY failure_rate_pct DESC;"""

failure_by_type = pd.read_sql(query,conn)
print("Failure Rate By Machine Type:\n")
print(failure_by_type.to_string(index=False))

Failure Rate By Machine Type:

machine_type  total_machines  total_failures  failure_rate_pct
           L            6000             235              3.92
           M            2997              83              2.77
           H            1003              21              2.09


## 4. Failure Type Analysis

Analyzing the distribution of specific failure types to understand which failures are most common.

In [8]:
#query 5 - most common failure types
query = """
SELECT 
    'Tool Wear Failure' as failure_type,
    SUM(tool_wear_failure) as count,
    ROUND(SUM(tool_wear_failure) * 100.0 / (SELECT SUM(machine_failure) FROM equipment_data), 2) as pct_of_failures
FROM equipment_data
UNION ALL
SELECT 
    'Heat Dissipation Failure',
    SUM(heat_dissipation_failure),
    ROUND(SUM(heat_dissipation_failure) * 100.0 / (SELECT SUM(machine_failure) FROM equipment_data), 2)
FROM equipment_data
UNION ALL
SELECT 
    'Power Failure',
    SUM(power_failure),
    ROUND(SUM(power_failure) * 100.0 / (SELECT SUM(machine_failure) FROM equipment_data), 2)
FROM equipment_data
UNION ALL
SELECT 
    'Overstrain Failure',
    SUM(overstrain_failure),
    ROUND(SUM(overstrain_failure) * 100.0 / (SELECT SUM(machine_failure) FROM equipment_data), 2)
FROM equipment_data
UNION ALL
SELECT 
    'Random Failure',
    SUM(random_failure),
    ROUND(SUM(random_failure) * 100.0 / (SELECT SUM(machine_failure) FROM equipment_data), 2)
FROM equipment_data
ORDER BY count DESC;
"""

failure_types = pd.read_sql(query, conn)
print('Failure Type Distribution:\n')
print(failure_types.to_string(index=False))

print('\nKey Insight:')
most_common = failure_types.iloc[0]
print(f'   "{most_common["failure_type"]}" is the most common at {most_common["count"]} occurrences ({most_common["pct_of_failures"]}% of all failures)')

Failure Type Distribution:

            failure_type  count  pct_of_failures
Heat Dissipation Failure    115            33.92
      Overstrain Failure     98            28.91
           Power Failure     95            28.02
       Tool Wear Failure     46            13.57
          Random Failure     19             5.60

Key Insight:
   "Heat Dissipation Failure" is the most common at 115 occurrences (33.92% of all failures)


In [9]:
# query 6 - failure type distribution by machine type
query = """
SELECT 
    machine_type,
    SUM(tool_wear_failure) as twf,
    SUM(heat_dissipation_failure) as hdf,
    SUM(power_failure) as pwf,
    SUM(overstrain_failure) as osf,
    SUM(random_failure) as rnf,
    SUM(machine_failure) as total_failures
FROM equipment_data
GROUP BY machine_type
ORDER BY total_failures DESC;
"""

failure_by_type_detail = pd.read_sql(query, conn)
print('Failure Types by Machine Type:\n')
print(failure_by_type_detail.to_string(index=False))

Failure Types by Machine Type:

machine_type  twf  hdf  pwf  osf  rnf  total_failures
           L   25   76   59   87   13             235
           M   14   31   31    9    2              83
           H    7    8    5    2    4              21



## 5. Sensor Reading Analysis

Comparing sensor readings between failed and non-failed machines to identify failure indicators.

### 5.1 Temperature Analysis

In [10]:
#query 7 - temperature comparison (failed vs non-failed)
query = """
SELECT 
    CASE 
        WHEN machine_failure = 0 THEN 'No Failure'
        WHEN machine_failure = 1 THEN 'Failure'
    END as status,
    ROUND(AVG(air_temp_k), 2) as avg_air_temp_k,
    ROUND(MIN(air_temp_k), 2) as min_air_temp_k,
    ROUND(MAX(air_temp_k), 2) as max_air_temp_k,
    ROUND(AVG(process_temp_k), 2) as avg_process_temp_k,
    ROUND(MIN(process_temp_k), 2) as min_process_temp_k,
    ROUND(MAX(process_temp_k), 2) as max_process_temp_k
FROM equipment_data
GROUP BY machine_failure;
"""

temp_analysis = pd.read_sql(query, conn)
print(' Temperature Analysis (Failed vs Non-Failed Machines):\n')
print(temp_analysis.to_string(index=False))

# Calculate temperature difference
air_temp_diff = abs(temp_analysis.loc[0, 'avg_air_temp_k'] - temp_analysis.loc[1, 'avg_air_temp_k'])
process_temp_diff = abs(temp_analysis.loc[0, 'avg_process_temp_k'] - temp_analysis.loc[1, 'avg_process_temp_k'])

print(f'\n Key Insight:')
print(f'Average air temperature difference: {air_temp_diff:.2f}K')
print(f'Average process temperature difference: {process_temp_diff:.2f}K')

 Temperature Analysis (Failed vs Non-Failed Machines):

    status  avg_air_temp_k  min_air_temp_k  max_air_temp_k  avg_process_temp_k  min_process_temp_k  max_process_temp_k
No Failure          299.97           295.3           304.5              310.00               305.7               313.8
   Failure          300.89           295.6           304.4              310.29               306.1               313.7

 Key Insight:
Average air temperature difference: 0.92K
Average process temperature difference: 0.29K


### 5.2 Rotational Speed Analysis

In [11]:
# Query 8: Rotational speed comparison
query = """
SELECT 
    CASE 
        WHEN machine_failure = 0 THEN 'No Failure'
        WHEN machine_failure = 1 THEN 'Failure'
    END as status,
    ROUND(AVG(rotational_speed_rpm), 2) as avg_speed_rpm,
    MIN(rotational_speed_rpm) as min_speed_rpm,
    MAX(rotational_speed_rpm) as max_speed_rpm,
    COUNT(*) as count
FROM equipment_data
GROUP BY machine_failure;
"""

speed_analysis = pd.read_sql(query, conn)
print('Rotational Speed Analysis (Failed vs Non-Failed Machines):\n')
print(speed_analysis.to_string(index=False))

speed_diff = abs(speed_analysis.loc[0, 'avg_speed_rpm'] - speed_analysis.loc[1, 'avg_speed_rpm'])
print(f'\n Key Insight:')
print(f'Average rotational speed difference: {speed_diff:.2f} RPM')

Rotational Speed Analysis (Failed vs Non-Failed Machines):

    status  avg_speed_rpm  min_speed_rpm  max_speed_rpm  count
No Failure        1540.26           1168           2695   9661
   Failure        1496.49           1181           2886    339

 Key Insight:
Average rotational speed difference: 43.77 RPM


### 5.3 Torque Analysis

In [12]:
#query 9 - Torque comparison
query = """
SELECT 
    CASE 
        WHEN machine_failure = 0 THEN 'No Failure'
        WHEN machine_failure = 1 THEN 'Failure'
    END as status,
    ROUND(AVG(torque_nm), 2) as avg_torque_nm,
    ROUND(MIN(torque_nm), 2) as min_torque_nm,
    ROUND(MAX(torque_nm), 2) as max_torque_nm,
    COUNT(*) as count
FROM equipment_data
GROUP BY machine_failure;
"""

torque_analysis = pd.read_sql(query, conn)
print(' Torque Analysis (Failed vs Non-Failed Machines):\n')
print(torque_analysis.to_string(index=False))

torque_diff = abs(torque_analysis.loc[0, 'avg_torque_nm'] - torque_analysis.loc[1, 'avg_torque_nm'])
print(f'\n Key Insight:')
print(f'   Average torque difference: {torque_diff:.2f} Nm')

 Torque Analysis (Failed vs Non-Failed Machines):

    status  avg_torque_nm  min_torque_nm  max_torque_nm  count
No Failure          39.63           12.6           70.0   9661
   Failure          50.17            3.8           76.6    339

 Key Insight:
   Average torque difference: 10.54 Nm


## 5.4 Tool Wear Analysis

In [13]:
#query 10 - Tool wear comparison
query = """
SELECT 
    CASE 
        WHEN machine_failure = 0 THEN 'No Failure'
        WHEN machine_failure = 1 THEN 'Failure'
    END as status,
    ROUND(AVG(tool_wear_min), 2) as avg_tool_wear_min,
    MIN(tool_wear_min) as min_tool_wear_min,
    MAX(tool_wear_min) as max_tool_wear_min,
    COUNT(*) as count
FROM equipment_data
GROUP BY machine_failure;
"""

tool_wear_analysis = pd.read_sql(query, conn)
print('Tool Wear Analysis (Failed vs Non-Failed Machines):\n')
print(tool_wear_analysis.to_string(index=False))

wear_diff = abs(tool_wear_analysis.loc[0, 'avg_tool_wear_min'] - tool_wear_analysis.loc[1, 'avg_tool_wear_min'])
print(f'\n Key Insight:')
print(f'Average tool wear difference: {wear_diff:.2f} minutes')

Tool Wear Analysis (Failed vs Non-Failed Machines):

    status  avg_tool_wear_min  min_tool_wear_min  max_tool_wear_min  count
No Failure             106.69                  0                246   9661
   Failure             143.78                  0                253    339

 Key Insight:
Average tool wear difference: 37.09 minutes


## 6.Advanced Patteren Analysis

In [14]:
#query 11 - High tool wear machine (potential risk)
query = """
SELECT
     machine_type,
     COUNT(*) AS machine_with_high_wear,
     SUM(machine_failure) AS failures,
     ROUND(AVG(machine_failure) * 100, 2) as failure_rate_pct
FROM equipment_data
WHERE tool_wear_min > 150
GROUP BY machine_type
ORDER BY failure_rate_pct DESC;
"""

high_wear = pd.read_sql(query,conn)
print('Machines with High Tool Wear (>150 min):\n')
print(high_wear.to_string(index=False))

Machines with High Tool Wear (>150 min):

machine_type  machine_with_high_wear  failures  failure_rate_pct
           L                    1842       139              7.55
           M                     897        35              3.90
           H                     299        10              3.34


Key Insight: 
Machines with high tool wear (>150 min) show elevated failure rates.

In [15]:
# query 12 - Extreme temperature conditions
query = """
SELECT 
    machine_type,
    COUNT(*) as extreme_temp_count,
    SUM(machine_failure) as failures,
    ROUND(AVG(machine_failure) * 100, 2) as failure_rate_pct
FROM equipment_data
WHERE process_temp_k > 310
GROUP BY machine_type
ORDER BY failure_rate_pct DESC;
"""

extreme_temp = pd.read_sql(query, conn)
print('Machines with Extreme Process Temperature (>310K):\n')
print(extreme_temp.to_string(index=False))

print('\n Key Insight:')
print('   High process temperatures (>310K) correlate with increased failure rates.')

Machines with Extreme Process Temperature (>310K):

machine_type  extreme_temp_count  failures  failure_rate_pct
           L                3070       153              4.98
           M                1495        59              3.95
           H                 472        11              2.33

 Key Insight:
   High process temperatures (>310K) correlate with increased failure rates.


Key Insight:
High process temperatures (>310K) correlate with increased failure rates.

In [16]:
# query 13 - Summary statistics by machine type
query = """
SELECT 
    machine_type,
    COUNT(*) as total_count,
    ROUND(AVG(air_temp_k), 2) as avg_air_temp,
    ROUND(AVG(process_temp_k), 2) as avg_process_temp,
    ROUND(AVG(rotational_speed_rpm), 2) as avg_speed,
    ROUND(AVG(torque_nm), 2) as avg_torque,
    ROUND(AVG(tool_wear_min), 2) as avg_tool_wear,
    ROUND(AVG(machine_failure) * 100, 2) as failure_rate_pct
FROM equipment_data
GROUP BY machine_type
ORDER BY failure_rate_pct DESC;
"""

summary_stats = pd.read_sql(query, conn)
print('Summary Statistics by Machine Type:\n')
print(summary_stats.to_string(index=False))

Summary Statistics by Machine Type:

machine_type  total_count  avg_air_temp  avg_process_temp  avg_speed  avg_torque  avg_tool_wear  failure_rate_pct
           L         6000        300.02            310.01    1539.47       40.00         108.38              3.92
           M         2997        300.03            310.02    1537.60       40.02         107.27              2.77
           H         1003        299.87            309.93    1538.15       39.84         107.42              2.09


In [17]:
# Close connection
conn.close()
print('Database connection closed successfully.')

Database connection closed successfully.


---

KEY FINDINGS FROM SQL ANALYSIS


FAILURE RATES BY MACHINE TYPE
   • Different machine types show varying failure rates  
   • Analyzed using GROUP BY and aggregation functions  

COMMON FAILURE TYPES
   • Tool Wear Failure, Heat Dissipation Failure, and Power Failure  
     are the most common failure modes  
   • Each failure type has distinct patterns  

SENSOR READING PATTERNS
   • Failed machines show different sensor readings compared to  
     non-failed machines  
   • Temperature, torque, and tool wear are key indicators  

HIGH-RISK CONDITIONS IDENTIFIED
   • Tool wear > 150 minutes → higher failure risk  
   • Process temperature > 310K → increased failures  
   • These conditions can trigger preventive maintenance  

---

